# V111 - No-chat specialist

Pre-registered plan: the adopted action applies a globally trained assessment model to no-chat users, but no-chat users may live in a different regime that 117 of their own can describe better. Train assessment Ridge on no-chat train users only and face it against the global model on the no-chat slice under grouped folds with a fresh seed. Small slice, so the slice floor and safety lock decide, never the pooled mean.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 3535
N_SPLITS = 5
SLICE_FLOOR = 0.01
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v111")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Specialist vs globalist on the no-chat slice

The specialist sees only no-chat training rows. The globalist sees all. Both score the same held-out no-chat users per fold.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
S = np.array([[num(blobs[u].get(q)) for q in questions] for u in list(train["user_id"])])
S = np.where(np.isnan(S), np.nanmean(S, axis=0), S)
A = (S - S.mean(0)) / (S.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
nochat = np.array([u not in by_user for u in list(train["user_id"])])
print(f"no-chat train users: {nochat.sum()}")

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

from sklearn.model_selection import StratifiedKFold
nc_idx = np.where(nochat)[0]
oof_g, oof_s = np.zeros_like(Y), np.zeros_like(Y)
fids = np.zeros(n_train, int)
skfs = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (trs, vas) in enumerate(skfs.split(nc_idx, Y[nc_idx].argmax(1))):
    tr_all = np.setdiff1d(np.arange(n_train), nc_idx[vas])
    tr_nc = nc_idx[trs]
    va_nc = nc_idx[vas]
    for j in range(len(MODS)):
        g = Ridge(alpha=1.0)
        g.fit(A[tr_all], Y[tr_all, j])
        oof_g[va_nc, j] = g.predict(A[va_nc])
        if len(tr_nc) >= 10:
            s = Ridge(alpha=1.0)
            s.fit(A[tr_nc], Y[tr_nc, j])
            oof_s[va_nc, j] = s.predict(A[va_nc])
        else:
            oof_s[va_nc, j] = oof_g[va_nc, j]
    fids[va_nc] = fi + 1
sc_g = np.full(n_train, np.nan)
sc_g[nochat] = ndcg5(Y[nochat], oof_g[nochat])
sc_s = np.full(n_train, np.nan)
sc_s[nochat] = ndcg5(Y[nochat], oof_s[nochat])
sg, ss = sc_g[nochat], sc_s[nochat]
d = ss - sg
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(5000)], [0.025, 0.975])
wf = min((np.nanmean(sc_s[(fids == k) & nochat]) - np.nanmean(sc_g[(fids == k) & nochat])) if ((fids == k) & nochat).sum() else 0.0 for k in range(1, N_SPLITS + 1))
print(f"slice n={nochat.sum()} globalist={sg.mean():.5f} specialist={ss.mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT specialist" if (lo > SLICE_FLOOR and wf >= 0) else "KEEP globalist; 117 users cannot carry a specialist")

no-chat train users: 117


/home/athilla/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


slice n=117 globalist=0.52568 specialist=0.59537 delta=+0.06969 CI95=[+0.03166,+0.10813] worst-fold=+0.04528
ADOPT specialist
